# 00 - Préparation (à exécuter une seule fois, runtime GPU)
Fine-tune la tête de ResNet-18 sur CIFAR-10 (10 classes) et sauvegarde les 50 images prétraitées, communes aux 4 pipelines.

In [4]:
import os

folder = '/content/drive/MyDrive/tp_deploy'

print(os.listdir(folder))

['results', 'common.py']


In [10]:
import os
import sys
import importlib.util

folder = '/content/drive/MyDrive/tp_deploy'

print("Fichier présent :", os.path.isfile(os.path.join(folder, 'common.py')))

if folder not in sys.path:
    sys.path.insert(0, folder)

print("Module détecté :", importlib.util.find_spec('common'))

Fichier présent : True
Module détecté : None


In [11]:
import importlib.util
import sys
import os

path = '/content/drive/MyDrive/tp_deploy/common.py'

spec = importlib.util.spec_from_file_location("common", path)
common = importlib.util.module_from_spec(spec)

sys.modules["common"] = common
spec.loader.exec_module(common)

print("Importation réussie !")

Importation réussie !


In [12]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os
os.makedirs('/content/drive/MyDrive/tp_deploy/results', exist_ok=True)
sys.path.append('/content/drive/MyDrive/tp_deploy')
from common import *
import torchvision
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets
tf = transforms.Compose([transforms.Resize(224), transforms.ToTensor(),
                         transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
train = datasets.CIFAR10('./data', train=True, download=True, transform=tf)
test = datasets.CIFAR10('./data', train=False, download=True, transform=tf)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


100%|██████████| 170M/170M [42:59<00:00, 66.1kB/s]


In [13]:
torch.manual_seed(0)
dev = 'cuda'
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = torch.nn.Linear(512, 10)
model.to(dev).train()
sub = Subset(train, np.random.RandomState(0).choice(50000, 10000, replace=False))
loader = DataLoader(sub, batch_size=64, shuffle=True, num_workers=2)
opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
for epoch in range(2):
    for xb, yb in loader:
        xb, yb = xb.to(dev), yb.to(dev)
        with torch.autocast('cuda'):
            loss = torch.nn.functional.cross_entropy(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
    print('epoch', epoch, 'loss', loss.item())
torch.save(model.cpu().state_dict(), f'{DRIVE}/resnet18_cifar10.pt')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 169MB/s]


epoch 0 loss 0.5597996711730957
epoch 1 loss 0.13114911317825317


In [14]:
idx = np.random.RandomState(42).choice(len(test), 50, replace=False)
Xs = torch.stack([test[int(i)][0] for i in idx])
ys = torch.tensor([test[int(i)][1] for i in idx])
torch.save({'X': Xs, 'y': ys, 'idx': idx.tolist()}, f'{DRIVE}/inputs.pt')
print(Xs.shape, ys.bincount(minlength=10))

torch.Size([50, 3, 224, 224]) tensor([4, 4, 7, 5, 3, 9, 5, 4, 5, 4])
